In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import KFold
import os
from pathlib import Path

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
class findPath():
    def __init__(self,pathname):
        self.pathname = pathname
            
    def parent(self):
        self.pathname = os.path.dirname(os.path.dirname(self.pathname))
        return findPath(self.pathname)
    
    def path(self,level=1):  
        for i in range(level):
            self.pathname = os.path.dirname(self.pathname)
        return self.pathname

In [ ]:
dataset_path = "benchmark/data/dataset1/"

In [6]:
findPath(dataset_path).path(1)

'/home/ctm/Documents/ML/MLTest/symaps_data_analyze/benchmark/data/dataset1'

In [7]:
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
import lib.dataGenLib as dg
import yaml
metadata = explore_csv_hierarchy('../benchmark/data',['dataset','set'])

data_list = ['new_dataset10','new_dataset10_const_pos_offset','new_dataset10_random_pos_offset_low'
             ,'new_dataset10_random_pos_offset_median','new_dataset10_random_pos_offset_high']
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',data_list),('set',data_list)])
metadata

,dataset,set,filename,path
12,new_dataset10_random_pos_offset_high,new_dataset10_random_pos_offset_high,training_0_999_20261002_14:36:30.csv,../benchmark/data/new_dataset10_random_pos_off...
43,new_dataset10_random_pos_offset_median,new_dataset10_random_pos_offset_median,training_0_999_20261002_14:36:30.csv,../benchmark/data/new_dataset10_random_pos_off...
70,new_dataset10_random_pos_offset_low,new_dataset10_random_pos_offset_low,training_0_999_20261002_14:36:30.csv,../benchmark/data/new_dataset10_random_pos_off...
171,new_dataset10,new_dataset10,training_0_999_20261002_14:36:30.csv,../benchmark/data/new_dataset10/new_dataset10/...


In [9]:
rolling_data_list = {}

for n,sample in metadata.iterrows():
    kf = KFold(n_splits=5)
    batchs = 5
    path = sample['path']
    raw_data = pd.read_csv(path)
    
    # regenerate rolling window dataset
    # stride 5s
    # rolling_window_df_1 = dg.rolling_window(raw_data.copy(),stride=50)
    # rolling_data_list['new_dataset10_r1'] = rolling_window_df_1
    # stride 1s
    rolling_window_df_2 = dg.rolling_window(raw_data.copy(),stride=10)
    rolling_data_list['new_dataset10_r2'] = rolling_window_df_2
    
    for name, rolling_data in rolling_data_list.items():
        dataset_path = str(findPath(path).parent().pathname) + '/' + name
        # print(dataset_path)
        rolling_data_path = dataset_path + '/'+ name + '.csv'
        print(rolling_data_path)
        Path(rolling_data_path).parent.mkdir(parents=True, exist_ok=True)
        rolling_data.to_csv(rolling_data_path,index=False)

print('finish')


../benchmark/data/new_dataset10_random_pos_offset_high/new_dataset10_r2/new_dataset10_r2.csv
../benchmark/data/new_dataset10_random_pos_offset_median/new_dataset10_r2/new_dataset10_r2.csv
../benchmark/data/new_dataset10_random_pos_offset_low/new_dataset10_r2/new_dataset10_r2.csv
../benchmark/data/new_dataset10/new_dataset10_r2/new_dataset10_r2.csv
finish


In [10]:
from abench.store.data_management import explore_csv_hierarchy,filter_metadata
import lib.dataGenLib as dg
import yaml
metadata = explore_csv_hierarchy('../benchmark/data',['dataset','set'])

data_list = ['new_dataset10','new_dataset10_const_pos_offset','new_dataset10_random_pos_offset_low'
             ,'new_dataset10_random_pos_offset_median','new_dataset10_random_pos_offset_high']
metadata=filter_metadata(metadata,constraint_selection_list=[('dataset',data_list),('set',['new_dataset10_r2'])])
metadata

,dataset,set,filename,path
13,new_dataset10_random_pos_offset_high,new_dataset10_r2,new_dataset10_r2.csv,../benchmark/data/new_dataset10_random_pos_off...
45,new_dataset10_random_pos_offset_median,new_dataset10_r2,new_dataset10_r2.csv,../benchmark/data/new_dataset10_random_pos_off...
72,new_dataset10_random_pos_offset_low,new_dataset10_r2,new_dataset10_r2.csv,../benchmark/data/new_dataset10_random_pos_off...
154,new_dataset10,new_dataset10_r2,new_dataset10_r2.csv,../benchmark/data/new_dataset10/new_dataset10_...


In [11]:
for n,sample in metadata.iterrows():
    kf = KFold(n_splits=5)
    batchs = 5
    path = sample['path']
    raw_data = pd.read_csv(path)

    row, col = raw_data.shape
    sequence_lenth = 50
    for i,(train_idx,test_idx) in enumerate(kf.split(range(0,row//sequence_lenth))):
        # print(f"Fold {i}")
        # print(f"  Train: index={train_idx},size={train_idx.shape}")
        # print(f"  Test:  index={test_idx},size={test_idx.shape}")
        for batch in range(batchs):
            start = batch*(test_idx[-1]-test_idx[0])//batchs + test_idx[0]
            # print(batch)
            if (batch+1) not in range(batchs):
                dataset = raw_data[start*sequence_lenth:].copy()
            else:
                end = (batch+1)*(test_idx[-1]-test_idx[0])//batchs - 1 + test_idx[0]
                # print(f"start:{start},,end:{end}")
                dataset = raw_data[start*sequence_lenth:end*sequence_lenth].copy()
            
            path_f = f"{findPath(path).path(2)}/r2_set_{i}/batch_{batch}.csv"
            # print(path_f)
            Path(path_f).parent.mkdir(parents=True, exist_ok=True)
            dataset.to_csv(path_f,index=False)
print('finish')

finish
